In [ ]:
# improved_no_okay_gesture_test_v3_fixed.py
import os, math, csv, json
import cv2
import numpy as np
from collections import defaultdict
from itertools import islice
from google.colab.patches import cv2_imshow  # For displaying images in Colab

# ---------- CONFIG ----------
BASE_DIR = '/content/drive/MyDrive/dataset'
OUT_CSV = '/content/gesture_results.csv'
ANNOTATED_DIR = '/content/annotatedimg'
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
RESIZE_WIDTH = 600

# Modes
DEBUG = True        # If True → shows masks and annotated results per image
DIAGNOSTIC = True   # Save per-class sample diagnostics (good for debugging)
SAMPLES_PER_CLASS = 6

os.makedirs(ANNOTATED_DIR, exist_ok=True)
os.makedirs(os.path.join(ANNOTATED_DIR, 'masks'), exist_ok=True)
os.makedirs(os.path.join(ANNOTATED_DIR, 'wrong_cases'), exist_ok=True)
os.makedirs(os.path.join(ANNOTATED_DIR, 'diagnostics'), exist_ok=True)

# ---------- UTIL ----------
def skin_mask(img):
    img_blur = cv2.GaussianBlur(img, (5, 5), 0)
    hsv = cv2.cvtColor(img_blur, cv2.COLOR_BGR2HSV)
    lower_hsv = np.array([0, 15, 60], dtype=np.uint8)
    upper_hsv = np.array([25, 255, 255], dtype=np.uint8)
    m1 = cv2.inRange(hsv, lower_hsv, upper_hsv)

    ycrcb = cv2.cvtColor(img_blur, cv2.COLOR_BGR2YCrCb)
    lower_ycrcb = np.array([0, 135, 85], dtype=np.uint8)
    upper_ycrcb = np.array([255, 180, 135], dtype=np.uint8)
    m2 = cv2.inRange(ycrcb, lower_ycrcb, upper_ycrcb)

    mask = cv2.bitwise_and(m1, m2)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.GaussianBlur(mask, (5, 5), 0)
    _, mask = cv2.threshold(mask, 60, 255, cv2.THRESH_BINARY)
    return mask

def nms_points(pts, min_dist=12):
    if pts is None or len(pts) == 0: return []
    pts = [tuple(map(int, p)) for p in pts]
    pts = sorted(pts, key=lambda p: (p[1], p[0]))
    kept = []
    for p in pts:
        if all(math.hypot(p[0]-q[0], p[1]-q[1]) >= min_dist for q in kept):
            kept.append(p)
    return kept

def angle_at_point(p_prev, p, p_next):
    v1 = (p_prev[0]-p[0], p_prev[1]-p[1])
    v2 = (p_next[0]-p[0], p_next[1]-p[1])
    a, b = math.hypot(*v1), math.hypot(*v2)
    if a == 0 or b == 0: return 180.0
    cosang = max(-1.0, min(1.0, (v1[0]*v2[0]+v1[1]*v2[1]) / (a*b)))
    return math.degrees(math.acos(cosang))

# ---------- DETECTION ----------
def detect_fingers(image):
    original = image.copy()
    h0, w0 = image.shape[:2]
    scale = RESIZE_WIDTH / float(w0) if w0 != RESIZE_WIDTH else 1.0
    if scale != 1.0:
        new_h = max(1, int(h0 * scale))
        image = cv2.resize(image, (RESIZE_WIDTH, new_h), interpolation=cv2.INTER_AREA)

    mask = skin_mask(image)
    mask_save = cv2.resize(mask, (w0, h0), interpolation=cv2.INTER_NEAREST)

    contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    stats = {}

    if not contours:
        stats['reason'] = 'no_contours'
        return original, "NOHAND", mask_save, stats

    cnt = max(contours, key=cv2.contourArea)
    cnt_area = cv2.contourArea(cnt)
    if cnt_area < 1500:
        stats.update({'reason': 'too_small', 'cnt_area': cnt_area})
        return original, "TOOSMALL", mask_save, stats

    cnt = cnt.reshape(-1, 2).astype(np.int32)
    x, y, w, h = cv2.boundingRect(cnt)
    bbox_diag = math.hypot(w, h)
    bbox_area = max(1, w * h)
    area_ratio = cnt_area / float(bbox_area)

    # Hulls
    hull_pts = cv2.convexHull(cnt.reshape(-1, 1, 2), returnPoints=True)
    hull_indices = cv2.convexHull(cnt.reshape(-1, 1, 2), returnPoints=False)
    hull_area = cv2.contourArea(hull_pts) if len(hull_pts) >= 3 else 0.0
    solidity = (cnt_area / hull_area) if hull_area > 0 else 0.0

    M = cv2.moments(cnt)
    cx, cy = (int(M['m10']/M['m00']), int(M['m01']/M['m00'])) if M['m00'] != 0 else (x + w//2, y + h//2)
    perimeter = cv2.arcLength(cnt.reshape(-1, 1, 2), True)
    circularity = (4 * math.pi * cnt_area) / (perimeter * perimeter + 1e-9)

    hull_pts_flat = [(int(p[0][0]), int(p[0][1])) for p in hull_pts]
    hull_candidates = []

    if len(hull_pts_flat) >= 3:
        dists = [math.hypot(px - cx, py - cy) for (px, py) in hull_pts_flat]
        maxd = max(dists)
        n = len(hull_pts_flat)
        for i, p in enumerate(hull_pts_flat):
            prev_p, next_p = hull_pts_flat[(i-1) % n], hull_pts_flat[(i+1) % n]
            ang = angle_at_point(prev_p, p, next_p)
            dist = dists[i]
            if ang < 60.0 and dist > 0.65 * maxd and p[1] < cy + 0.12 * h:
                hull_candidates.append((p, ang, dist))

    hull_candidates_pts = [p for p, _, _ in sorted(hull_candidates, key=lambda x: -x[2])]
    hull_fingertips = nms_points(hull_candidates_pts, min_dist=max(10, int(0.028 * max(w, h))))

    # Defects
    defects = cv2.convexityDefects(cnt.astype(np.int32), hull_indices.astype(np.int32)) if len(hull_indices) > 3 else None
    defect_fars = []
    defect_count = 0

    if defects is not None:
        depth_thresh_px = max(10.0, 0.06 * bbox_diag)
        for i in range(defects.shape[0]):
            s, e, f, d = defects[i, 0]
            start, end, far = tuple(cnt[s]), tuple(cnt[e]), tuple(cnt[f])
            a, b, c = math.hypot(end[0]-start[0], end[1]-start[1]), math.hypot(far[0]-start[0], far[1]-start[1]), math.hypot(end[0]-far[0], end[1]-far[1])
            if b == 0 or c == 0: continue
            cos_angle = max(-1.0, min(1.0, (b*b + c*c - a*a) / (2*b*c)))
            angle = math.degrees(math.acos(cos_angle))
            depth = d / 256.0
            if angle < 75 and depth > depth_thresh_px and far[1] < cy + 0.2 * h:
                defect_count += 1
                defect_fars.append(far)
    defect_fars = nms_points(defect_fars, min_dist=max(10, int(0.028 * max(w, h))))

    # Fingertip decision
    final_fingertips = hull_fingertips if len(hull_fingertips) > 0 else defect_fars
    finger_count_est = min(5, len(final_fingertips)) if final_fingertips else (min(5, defect_count + 1) if defect_count > 0 else 0)

    # FIST heuristic
    is_compact = (area_ratio > 0.55 and circularity < 0.35)
    fist_by_solidity = (solidity > 0.90 and defect_count == 0)
    fist_by_compact = (is_compact and finger_count_est <= 1)
    if fist_by_solidity or fist_by_compact:
        predicted_count = 0
    else:
        predicted_count = finger_count_est

    """
    if predicted_count == 0:
        aspect = h / (w + 1e-9)
        label = "FIST" if (solidity > 0.80 and aspect < 1.0) or fist_by_compact or fist_by_solidity else "ONE"
    else:
        label = {1: 'ONE', 2: 'TWO', 3: 'THREE', 4: 'FOUR', 5: 'FIVE'}.get(predicted_count, 'FIVE')
    """

    # Label decision based on defects and solidity
    if defect_count == 0 and solidity > 0.9:
        label = "FIST"
    elif defect_count == 0 and solidity <= 0.9:
        label = "ONE"
    elif defect_count == 1:
        label = "TWO"
    elif defect_count == 2:
        label = "THREE"
    elif defect_count == 3:
        label = "FOUR"
    elif defect_count >= 4:
        label = "FIVE"
    else:
        label = "UNKNOWN"




    # Annotations
    vis = image.copy()
    cv2.rectangle(vis, (x, y), (x + w, y + h), (255, 0, 0), 2)
    cv2.drawContours(vis, [cnt.reshape(-1, 1, 2)], -1, (0, 255, 0), 2)
    for p in hull_pts_flat: cv2.circle(vis, p, 3, (0, 0, 255), -1)
    for p in hull_candidates_pts: cv2.circle(vis, p, 4, (0, 165, 255), 1)
    for p in hull_fingertips: cv2.circle(vis, p, 8, (0, 255, 255), -1)
    for p in defect_fars: cv2.circle(vis, p, 5, (255, 255, 255), -1)
    cv2.circle(vis, (cx, cy), 4, (255, 0, 255), -1)
    cv2.putText(vis, f"{label}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

    annotated_full = cv2.resize(vis, (w0, h0), interpolation=cv2.INTER_LINEAR) if scale != 1.0 else vis

    if DEBUG:
        print("STATS:", json.dumps({'label': label, 'area': cnt_area, 'solidity': solidity, 'defects': defect_count}))
        #cv2_imshow(mask)
        #cv2_imshow(annotated_full)

    return annotated_full, label, mask_save, {}

# ---------- I/O & EVAL ----------
def all_image_paths(base_dir):
    for folder in sorted(os.listdir(base_dir)):
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path): continue
        for fname in sorted(os.listdir(folder_path)):
            if os.path.splitext(fname)[1].lower() in IMG_EXTS:
                yield folder, os.path.join(folder_path, fname)

def evaluate(base_dir, out_csv, annotated_dir):
    classes_all = sorted([f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))])
    classes = [c for c in classes_all if c.strip().upper() != 'OKAY']
    per_class = {c: {'total': 0, 'correct': 0} for c in classes}
    results, wrong = [], []
    diagnostics_store = defaultdict(list)

    for folder, img_path in all_image_paths(base_dir):
        if folder.strip().upper() == 'OKAY': continue
        gt_label = folder.strip().upper()
        img = cv2.imread(img_path)
        if img is None:
            print("Warning: cannot read", img_path)
            continue

        annotated_img, pred_label, mask_img, stats = detect_fingers(img)

        save_class_dir = os.path.join(annotated_dir, folder)
        os.makedirs(save_class_dir, exist_ok=True)
        cv2.imwrite(os.path.join(save_class_dir, os.path.basename(img_path)), annotated_img)
        mask_dir = os.path.join(annotated_dir, 'masks', folder)
        os.makedirs(mask_dir, exist_ok=True)
        cv2.imwrite(os.path.join(mask_dir, os.path.basename(img_path)), mask_img)

        results.append((img_path, gt_label, pred_label))
        per_class.setdefault(gt_label, {'total': 0, 'correct': 0})
        per_class[gt_label]['total'] += 1
        if gt_label == pred_label:
            per_class[gt_label]['correct'] += 1
        else:
            wrong.append((img_path, gt_label, pred_label))
            bad_dir = os.path.join(annotated_dir, 'wrong_cases')
            os.makedirs(bad_dir, exist_ok=True)
            cv2.imwrite(os.path.join(bad_dir, os.path.basename(img_path)), annotated_img)

    with open(out_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['image_path', 'ground_truth', 'predicted'])
        writer.writerows(results)

    total = sum(p['total'] for p in per_class.values())
    correct_total = sum(p['correct'] for p in per_class.values())
    overall_acc = (correct_total / total * 100) if total > 0 else 0

    print("\n=== SUMMARY ===")
    print(f"Overall accuracy: {overall_acc:.2f}% ({correct_total}/{total})")
    for c in classes:
        t, corr = per_class[c]['total'], per_class[c]['correct']
        acc = (corr / t * 100) if t > 0 else 0
        print(f"  {c}: {acc:.2f}% ({corr}/{t})")
    print(f"\nAnnotated images saved in: {annotated_dir}")
    print(f"CSV saved at: {out_csv}")

    return {
        'total': total,
        'correct_total': correct_total,
        'overall_accuracy': overall_acc,
        'wrong_list': wrong,
        'csv': out_csv,
        'annotated_dir': annotated_dir
    }

if _name_ == "_main_":
    stats = evaluate(BASE_DIR, OUT_CSV, ANNOTATED_DIR)

In [ ]:
# improved_no_okay_gesture_test_v3_fixed_fist_five_fix_v2.py
# Small patch to previous file: improved, scale-aware FIST / FIVE logic.

import os, math, csv, json
import cv2
import numpy as np
from collections import defaultdict
from itertools import islice
from google.colab.patches import cv2_imshow  # For displaying images in Colab

# ---------- CONFIG ----------
BASE_DIR = '/content/drive/MyDrive/dataset'
OUT_CSV = '/content/gesture_results.csv'
ANNOTATED_DIR = '/content/annotatedimg'
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
RESIZE_WIDTH = 600

# Modes
DEBUG = True
DIAGNOSTIC = True
SAMPLES_PER_CLASS = 6

os.makedirs(ANNOTATED_DIR, exist_ok=True)
os.makedirs(os.path.join(ANNOTATED_DIR, 'masks'), exist_ok=True)
os.makedirs(os.path.join(ANNOTATED_DIR, 'wrong_cases'), exist_ok=True)
os.makedirs(os.path.join(ANNOTATED_DIR, 'diagnostics'), exist_ok=True)

# ---------- UTIL ----------
def skin_mask(img):
    img_blur = cv2.GaussianBlur(img, (5, 5), 0)
    hsv = cv2.cvtColor(img_blur, cv2.COLOR_BGR2HSV)
    lower_hsv = np.array([0, 15, 60], dtype=np.uint8)
    upper_hsv = np.array([25, 255, 255], dtype=np.uint8)
    m1 = cv2.inRange(hsv, lower_hsv, upper_hsv)

    ycrcb = cv2.cvtColor(img_blur, cv2.COLOR_BGR2YCrCb)
    lower_ycrcb = np.array([0, 135, 85], dtype=np.uint8)
    upper_ycrcb = np.array([255, 180, 135], dtype=np.uint8)
    m2 = cv2.inRange(ycrcb, lower_ycrcb, upper_ycrcb)

    mask = cv2.bitwise_and(m1, m2)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.GaussianBlur(mask, (5, 5), 0)
    _, mask = cv2.threshold(mask, 60, 255, cv2.THRESH_BINARY)
    return mask

def nms_points(pts, min_dist=12):
    if pts is None or len(pts) == 0: return []
    pts = [tuple(map(int, p)) for p in pts]
    pts = sorted(pts, key=lambda p: (p[1], p[0]))
    kept = []
    for p in pts:
        if all(math.hypot(p[0]-q[0], p[1]-q[1]) >= min_dist for q in kept):
            kept.append(p)
    return kept

def angle_at_point(p_prev, p, p_next):
    v1 = (p_prev[0]-p[0], p_prev[1]-p[1])
    v2 = (p_next[0]-p[0], p_next[1]-p[1])
    a, b = math.hypot(*v1), math.hypot(*v2)
    if a == 0 or b == 0: return 180.0
    cosang = max(-1.0, min(1.0, (v1[0]*v2[0]+v1[1]*v2[1]) / (a*b)))
    return math.degrees(math.acos(cosang))

# ---------- DETECTION ----------
def detect_fingers(image):
    original = image.copy()
    h0, w0 = image.shape[:2]
    area_img = float(h0 * w0)
    scale = RESIZE_WIDTH / float(w0) if w0 != RESIZE_WIDTH else 1.0
    if scale != 1.0:
        new_h = max(1, int(h0 * scale))
        image = cv2.resize(image, (RESIZE_WIDTH, new_h), interpolation=cv2.INTER_AREA)

    mask = skin_mask(image)
    mask_save = cv2.resize(mask, (w0, h0), interpolation=cv2.INTER_NEAREST)

    contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    stats = {}

    if not contours:
        stats['reason'] = 'no_contours'
        return original, "NOHAND", mask_save, stats

    cnt = max(contours, key=cv2.contourArea)
    cnt_area = float(cv2.contourArea(cnt))
    if cnt_area < 1500:
        stats.update({'reason': 'too_small', 'cnt_area': cnt_area})
        return original, "TOOSMALL", mask_save, stats

    cnt = cnt.reshape(-1, 2).astype(np.int32)
    x, y, w, h = cv2.boundingRect(cnt)
    bbox_diag = math.hypot(w, h)
    bbox_area = float(max(1, w * h))
    area_ratio = cnt_area / float(bbox_area)

    # Hulls
    hull_pts = cv2.convexHull(cnt.reshape(-1, 1, 2), returnPoints=True)
    hull_indices = cv2.convexHull(cnt.reshape(-1, 1, 2), returnPoints=False)
    hull_area = float(cv2.contourArea(hull_pts)) if len(hull_pts) >= 3 else 0.0
    solidity = (cnt_area / hull_area) if hull_area > 0 else 0.0

    # centroid & geometry
    M = cv2.moments(cnt)
    cx, cy = (int(M['m10']/M['m00']), int(M['m01']/M['m00'])) if M['m00'] != 0 else (x + w//2, y + h//2)
    perimeter = cv2.arcLength(cnt.reshape(-1, 1, 2), True)
    circularity = (4 * math.pi * cnt_area) / (perimeter * perimeter + 1e-9)

    hull_pts_flat = [(int(p[0][0]), int(p[0][1])) for p in hull_pts]
    hull_candidates = []

    if len(hull_pts_flat) >= 3:
        dists = [math.hypot(px - cx, py - cy) for (px, py) in hull_pts_flat]
        maxd = max(dists)
        n = len(hull_pts_flat)
        for i, p in enumerate(hull_pts_flat):
            prev_p, next_p = hull_pts_flat[(i-1) % n], hull_pts_flat[(i+1) % n]
            ang = angle_at_point(prev_p, p, next_p)
            dist = dists[i]
            if ang < 70.0 and dist > 0.55 * maxd and p[1] < cy + 0.30 * h:
                hull_candidates.append((p, ang, dist))

    hull_candidates_pts = [p for p, _, _ in sorted(hull_candidates, key=lambda x: -x[2])]
    hull_fingertips = nms_points(hull_candidates_pts, min_dist=max(10, int(0.028 * max(w, h))))

    # Defects
    defects = cv2.convexityDefects(cnt.astype(np.int32), hull_indices.astype(np.int32)) if len(hull_indices) > 3 else None
    defect_fars = []
    defect_count = 0

    if defects is not None:
        depth_thresh_px = max(8.0, 0.05 * bbox_diag)
        for i in range(defects.shape[0]):
            s, e, f, d = defects[i, 0]
            start, end, far = tuple(cnt[s]), tuple(cnt[e]), tuple(cnt[f])
            a = math.hypot(end[0]-start[0], end[1]-start[1])
            b = math.hypot(far[0]-start[0], far[1]-start[1])
            c = math.hypot(end[0]-far[0], end[1]-far[1])
            if b == 0 or c == 0: continue
            cos_angle = max(-1.0, min(1.0, (b*b + c*c - a*a) / (2*b*c)))
            angle = math.degrees(math.acos(cos_angle))
            depth = d / 256.0
            if angle < 95 and depth > depth_thresh_px and far[1] < cy + 0.25 * h:
                defect_count += 1
                defect_fars.append(far)
    defect_fars = nms_points(defect_fars, min_dist=max(10, int(0.028 * max(w, h))))

    # Fingertip fusion
    combined_candidates = []
    combined_candidates.extend(hull_fingertips)
    combined_candidates.extend(defect_fars)   # weak candidates
    final_fingertips = nms_points(combined_candidates, min_dist=max(12, int(0.03 * max(w, h))))
    if len(final_fingertips) > 5:
        final_fingertips = final_fingertips[:5]
    fingertip_count = len(final_fingertips)

    # area-gap (absolute and relative)
    abs_gap = max(0.0, hull_area - cnt_area)                # absolute pixel gap
    arearatio_percent = ((hull_area - cnt_area) / (cnt_area + 1e-9)) * 100.0

    # conservative fused finger estimate
    est_by_fingertips = fingertip_count
    est_by_defects = defect_count + (1 if defect_count > 0 else 0)
    finger_count_est = max(est_by_fingertips, est_by_defects)
    finger_count_est = min(5, finger_count_est)

    # --------------------- NEW SAFER DECISION RULES ---------------------
    # thresholds (scale-aware)
    min_contour_fraction_for_gap_rule = 0.01   # contour must occupy >=1% of image to use gap-only rule
    min_contour_area_for_gap_rule = 0.01 * area_img   # absolute minimum contour area to consider areagap-based FIVE
    absolute_gap_thresh = 0.12 * bbox_area    # hull-contour gap must be >12% of bbox area to be meaningful
    relative_gap_percent_thresh = 35.0        # percent-based fallback (still require contour not tiny)

    # conservative FIST condition
    fist_by_solidity = (solidity >= 0.94 and defect_count == 0 and finger_count_est <= 1)
    compact_by_circularity = (circularity > 0.60 and finger_count_est <= 1)
    fist_by_small_gap = (abs_gap < 0.12 * bbox_area and finger_count_est <= 1 and cnt_area > 0)
    is_fist = (fist_by_solidity or compact_by_circularity or fist_by_small_gap)

    if is_fist:
        label = "FIST"
    else:
        # Conservative FIVE conditions:
        cond_defects_and_peaks = (defect_count >= 3 and fingertip_count >= 3)
        cond_abs_gap = (abs_gap > absolute_gap_thresh and cnt_area >= min_contour_area_for_gap_rule)
        cond_rel_gap = (arearatio_percent > relative_gap_percent_thresh and cnt_area >= min_contour_area_for_gap_rule)
        cond_many_peaks = (fingertip_count >= 4)
        if cond_defects_and_peaks or cond_abs_gap or (cond_rel_gap and cond_many_peaks) or cond_many_peaks:
            label = "FIVE"
        else:
            # fallback to defect-count mapping (safer)
            if defect_count == 0:
                # no defects -> rely on peaks
                if finger_count_est == 0:
                    label = "ONE"
                else:
                    label = {1: 'ONE', 2: 'TWO', 3: 'THREE', 4: 'FOUR', 5: 'FIVE'}.get(finger_count_est, 'ONE')
            elif defect_count == 1:
                label = "TWO"
            elif defect_count == 2:
                label = "THREE"
            elif defect_count == 3:
                label = "FOUR"
            elif defect_count >= 4:
                label = "FIVE"
            else:
                label = "UNKNOWN"
    # ------------------------------------------------------------------

    # Annotations
    vis = image.copy()
    cv2.rectangle(vis, (x, y), (x + w, y + h), (255, 0, 0), 2)
    cv2.drawContours(vis, [cnt.reshape(-1, 1, 2)], -1, (0, 255, 0), 2)
    for p in hull_pts_flat: cv2.circle(vis, p, 3, (0, 0, 255), -1)
    for p in hull_candidates_pts: cv2.circle(vis, p, 4, (0, 165, 255), 1)
    for p in final_fingertips: cv2.circle(vis, p, 8, (0, 255, 255), -1)
    for p in defect_fars: cv2.circle(vis, p, 5, (255, 255, 255), -1)
    cv2.circle(vis, (cx, cy), 4, (255, 0, 255), -1)
    cv2.putText(vis, f"{label}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

    annotated_full = cv2.resize(vis, (w0, h0), interpolation=cv2.INTER_LINEAR) if scale != 1.0 else vis

    stats_out = {
        'label': label,
        'area': float(cnt_area),
        'solidity': float(solidity),
        'defects': int(defect_count),
        'fingertips': int(fingertip_count),
        'arearatio_percent': float(arearatio_percent),
        'abs_gap': float(abs_gap),
        'bbox_area': float(bbox_area),
        'image_area': float(area_img),
        'circularity': float(circularity)
    }

    if DEBUG:
        print("STATS:", json.dumps(stats_out))

    return annotated_full, label, mask_save, stats_out

# ---------- I/O & EVAL ----------
def all_image_paths(base_dir):
    for folder in sorted(os.listdir(base_dir)):
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path): continue
        for fname in sorted(os.listdir(folder_path)):
            if os.path.splitext(fname)[1].lower() in IMG_EXTS:
                yield folder, os.path.join(folder_path, fname)

def evaluate(base_dir, out_csv, annotated_dir):
    classes_all = sorted([f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))])
    classes = [c for c in classes_all if c.strip().upper() != 'OKAY']
    per_class = {c: {'total': 0, 'correct': 0} for c in classes}
    results, wrong = [], []
    diagnostics_store = defaultdict(list)

    for folder, img_path in all_image_paths(base_dir):
        if folder.strip().upper() == 'OKAY': continue
        gt_label = folder.strip().upper()
        img = cv2.imread(img_path)
        if img is None:
            print("Warning: cannot read", img_path)
            continue

        annotated_img, pred_label, mask_img, stats = detect_fingers(img)

        save_class_dir = os.path.join(annotated_dir, folder)
        os.makedirs(save_class_dir, exist_ok=True)
        cv2.imwrite(os.path.join(save_class_dir, os.path.basename(img_path)), annotated_img)
        mask_dir = os.path.join(annotated_dir, 'masks', folder)
        os.makedirs(mask_dir, exist_ok=True)
        cv2.imwrite(os.path.join(mask_dir, os.path.basename(img_path)), mask_img)

        results.append((img_path, gt_label, pred_label))
        per_class.setdefault(gt_label, {'total': 0, 'correct': 0})
        per_class[gt_label]['total'] += 1
        if gt_label == pred_label:
            per_class[gt_label]['correct'] += 1
        else:
            wrong.append((img_path, gt_label, pred_label))
            bad_dir = os.path.join(annotated_dir, 'wrong_cases')
            os.makedirs(bad_dir, exist_ok=True)
            cv2.imwrite(os.path.join(bad_dir, os.path.basename(img_path)), annotated_img)

    with open(out_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['image_path', 'ground_truth', 'predicted'])
        writer.writerows(results)

    total = sum(p['total'] for p in per_class.values())
    correct_total = sum(p['correct'] for p in per_class.values())
    overall_acc = (correct_total / total * 100) if total > 0 else 0

    print("\n=== SUMMARY ===")
    print(f"Overall accuracy: {overall_acc:.2f}% ({correct_total}/{total})")
    for c in classes:
        t, corr = per_class[c]['total'], per_class[c]['correct']
        acc = (corr / t * 100) if t > 0 else 0
        print(f"  {c}: {acc:.2f}% ({corr}/{t})")
    print(f"\nAnnotated images saved in: {annotated_dir}")
    print(f"CSV saved at: {out_csv}")

    return {
        'total': total,
        'correct_total': correct_total,
        'overall_accuracy': overall_acc,
        'wrong_list': wrong,
        'csv': out_csv,
        'annotated_dir': annotated_dir
    }

if _name_ == "_main_":
    stats = evaluate(BASE_DIR, OUT_CSV, ANNOTATED_DIR)